In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import pandas as pd
import pickle
from src.sparse_dataloader import TrafficDataset

In [3]:
# Load processed weather data
processed_data_dir = 'data/datos_procesados'
sparse_data_dir = f'{processed_data_dir}/sparse_datasets'

# Load stations mapping
with open(f'{sparse_data_dir}/weather_stations.pkl', 'rb') as f:
    stations = pickle.load(f)

# Load processed weather data
wind_processed = pd.read_parquet(f'{sparse_data_dir}/weather_wind_direction.parquet')
humidity_processed = pd.read_parquet(f'{sparse_data_dir}/weather_humidity.parquet')

print(f"Stations loaded: {len(stations)}")
print(f"Wind data: {len(wind_processed)} records")
print(f"Humidity data: {len(humidity_processed)} records")

Stations loaded: 139107
Wind data: 487475 records
Humidity data: 1626201 records


In [4]:
# Load raw weather data for comparison
wind_raw = pd.read_parquet(f'{processed_data_dir}/weather/direccion_viento.parquet')
humidity_raw = pd.read_parquet(f'{processed_data_dir}/weather/humedad.parquet')

print("Processed wind sample:")
print(wind_processed.head())
print("\nRaw wind sample:")
print(wind_raw.head())

Processed wind sample:
   timestamp estacion_direccion_viento  direccion_viento  time_idx
0 2024-01-01           Cerro Cazadores        211.704667         0
1 2024-01-01                   Vitelma        110.366667         0
2 2024-01-01           UAN Circunvalar        137.840000         0
3 2024-01-01                  Juan Rey        179.460000         0
4 2024-01-01                Doña Juana        149.666667         0

Raw wind sample:
                       estacion_direccion_viento  direccion_viento
timestamp                                                         
2024-01-01                       Cerro Cazadores        211.704667
2024-01-01                      Chicu Río Bogotá        128.345333
2024-01-01  Colegio Alberto Lleras Camargo (IED)        199.166667
2024-01-01            Colegio El Manantial (IED)        179.033333
2024-01-01  Colegio Gabriel Garcia Márquez (IED)        173.093333


In [5]:
# Pick a specific station and timestamp for verification
test_station = 'Cerro Cazadores'
test_timestamp = pd.Timestamp('2024-01-01 00:00:00')

# Find in processed data
processed_sample = wind_processed[
    (wind_processed['estacion_direccion_viento'] == test_station) & 
    (wind_processed['timestamp'] == test_timestamp)
]

# Find in raw data  
raw_sample = wind_raw[
    (wind_raw['estacion_direccion_viento'] == test_station) & 
    (wind_raw.index == test_timestamp)
]

print(f"Processed value: {processed_sample['direccion_viento'].values[0] if len(processed_sample) > 0 else 'Not found'}")
print(f"Raw value: {raw_sample['direccion_viento'].values[0] if len(raw_sample) > 0 else 'Not found'}")
print(f"Match: {processed_sample['direccion_viento'].values[0] == raw_sample['direccion_viento'].values[0] if len(processed_sample) > 0 and len(raw_sample) > 0 else 'Cannot compare'}")

Processed value: 211.70466666666667
Raw value: 211.70466666666667
Match: True


In [6]:
# Test TrafficDataset with weather features
graph_filename = f'{processed_data_dir}/conectivity_graph/road_graph.pickle'
static_data_filename = f'{sparse_data_dir}/static_dataset.h5'
target_data_filename = f'{sparse_data_dir}/jams_dataset.h5'

dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=[f'{sparse_data_dir}/coi_points_dataset.h5'],
    metadata_paths=[f'{sparse_data_dir}/coi_points_dataset_metadata.pkl'],
    target_data_path=target_data_filename,
    in_timesteps=3,
    out_timesteps=2,
    future_features=['coi_puntos'],
    node_batch_size=5,
    weather_data_dir=sparse_data_dir,
    include_weather=True
)

sample = dataset[0]
print(f"Weather tensor shape: {sample.x_weather.shape}")
print(f"Weather values sample: {sample.x_weather[0, 0, :]}")

Weather tensor shape: torch.Size([5, 5, 4])
Weather values sample: tensor([277.9000,  88.7667,   0.4859,   0.0000])


In [7]:
# Verify specific node weather data matches raw data
node_id = dataset.id_to_node[0]  # First node in batch
station_wind = stations[node_id]['estacion_direccion_viento']
station_humidity = stations[node_id]['estacion_humedad']

print(f"Node {node_id} uses wind station: {station_wind}")
print(f"Node {node_id} uses humidity station: {station_humidity}")

# Check if tensor values match raw data for first timestep
tensor_wind = sample.x_weather[0, 0, 0].item()  # Wind direction
tensor_humidity = sample.x_weather[0, 0, 1].item()  # Humidity

print(f"Tensor wind value: {tensor_wind}")
print(f"Tensor humidity value: {tensor_humidity}")

# Find matching raw data for comparison
raw_wind_match = wind_raw[
    (wind_raw['estacion_direccion_viento'] == station_wind) & 
    (wind_raw.index == pd.Timestamp('2024-01-01 00:00:00'))
]
raw_humidity_match = humidity_raw[
    (humidity_raw['estacion_humedad'] == station_humidity) & 
    (humidity_raw.index == pd.Timestamp('2024-01-01 00:00:00'))
]

if len(raw_wind_match) > 0:
    raw_wind_value = raw_wind_match['direccion_viento'].values[0]
    print(f"Raw wind value: {raw_wind_value}")
    print(f"Wind values match: {tensor_wind == raw_wind_value}")
else:
    print("No raw wind data found for comparison")

if len(raw_humidity_match) > 0:
    raw_humidity_value = raw_humidity_match['humedad'].values[0]
    print(f"Raw humidity value: {raw_humidity_value}")
    print(f"Humidity values match: {tensor_humidity == raw_humidity_value}")
else:
    print("No raw humidity data found for comparison")

Node 0_B.1 uses wind station: Vitelma
Node 0_B.1 uses humidity station: Colegio Manuel Elkin Patarroyo (IED)
Tensor wind value: 277.8999938964844
Tensor humidity value: 88.76667022705078
Raw wind value: 110.36666666666666
Wind values match: False
Raw humidity value: 81.88000000000001
Humidity values match: False


In [ ]:
# FIXED VERIFICATION: Check the correct node mapping
# The issue was that we were checking dataset.id_to_node[0] but the actual batch 
# for dataset[0] contains different nodes based on the node_seed

# Get the actual nodes in the first batch
time_idx, node_indices = dataset._get_time_and_node_indices(0)
print(f"First batch contains nodes with indices: {node_indices}")
print(f"First batch contains node IDs: {[dataset.id_to_node[i] for i in node_indices]}")

# Check the first node in the actual batch (batch position 0)
actual_first_node_idx = node_indices[0]  # First node in batch
actual_first_node_id = dataset.id_to_node[actual_first_node_idx]
print(f"\nActual first node in batch: {actual_first_node_id}")

# Get weather stations for this node
station_wind = stations[actual_first_node_id]['estacion_direccion_viento']
station_humidity = stations[actual_first_node_id]['estacion_humedad']

print(f"Node {actual_first_node_id} uses wind station: {station_wind}")
print(f"Node {actual_first_node_id} uses humidity station: {station_humidity}")

# Check tensor values for batch position 0 (which corresponds to actual_first_node_id)
tensor_wind = sample.x_weather[0, 0, 0].item()  # Wind direction
tensor_humidity = sample.x_weather[0, 0, 1].item()  # Humidity

print(f"Tensor wind value: {tensor_wind}")
print(f"Tensor humidity value: {tensor_humidity}")

# Find matching raw data for comparison (using the CORRECT node and station)
raw_wind_match = wind_raw[
    (wind_raw['estacion_direccion_viento'] == station_wind) & 
    (wind_raw.index == pd.Timestamp('2024-01-01 00:00:00'))
]
raw_humidity_match = humidity_raw[
    (humidity_raw['estacion_humedad'] == station_humidity) & 
    (humidity_raw.index == pd.Timestamp('2024-01-01 00:00:00'))
]

if len(raw_wind_match) > 0:
    raw_wind_value = raw_wind_match['direccion_viento'].values[0]
    print(f"Raw wind value: {raw_wind_value}")
    print(f"Wind values match: {abs(tensor_wind - raw_wind_value) < 0.01}")  # Allow small floating point differences
else:
    print("No raw wind data found for comparison")

if len(raw_humidity_match) > 0:
    raw_humidity_value = raw_humidity_match['humedad'].values[0]
    print(f"Raw humidity value: {raw_humidity_value}")
    print(f"Humidity values match: {abs(tensor_humidity - raw_humidity_value) < 0.01}")  # Allow small floating point differences
else:
    print("No raw humidity data found for comparison")

First batch contains nodes with indices: [34, 6, 35, 7, 31]
First batch contains node IDs: ['0_FT.5', '0_FT.10', '0_FT.6', '0_FT.11', '0_FT.33']

Actual first node in batch: 0_FT.5
Node 0_FT.5 uses wind station: El Tabaco Río Bogotá
Node 0_FT.5 uses humidity station: Colegio Inem Francisco De Paula Santander (IED)
Tensor wind value: 277.8999938964844
Tensor humidity value: 88.76667022705078
Raw wind value: 277.9
Wind values match: True
Raw humidity value: 88.76666666666667
Humidity values match: True

CONCLUSION: The weather loading is working correctly!
The original verification was comparing the wrong nodes.
